# v1 OD流线图（差值模式）
基于参考代码 4-出租车OD.ipynb 的逐条绘制思路重写 flowline 函数

In [ ]:
import sys, os
sys.path.insert(0, r'E:/00_Commute_Scenario_Research')
os.chdir(r'E:/00_Commute_Scenario_Research')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from shapely.ops import unary_union
from pathlib import Path

from src.config import SHP_PATH, RESULTS_DIR, VISUAL_CONFIG, MAP_ELEMENTS
from src.visualization import add_north_arrow, _add_scalebar_auto

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

OUT_DIR = Path(r'E:/00_Commute_Scenario_Research/results/tempt/0418ODvis_tempt')
OUT_DIR.mkdir(parents=True, exist_ok=True)

fence = gpd.read_file(str(SHP_PATH))
print('fence shape:', fence.shape)


In [ ]:
# 准备差值 OD 数据
def load_od(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [c.lstrip('\ufeff') for c in df.columns]
    df = df.rename(columns={'Htaz': 'o', 'Jtaz': 'd', 'flow': '人数', 'distance_m': 'distance'})
    return df

od_actual   = load_od(RESULTS_DIR / '1.Data_Preprocess/实际格局-统一结构.csv')
od_baseline = load_od(RESULTS_DIR / '2.Pattern_Computation/2.4Baseline_Int/star_baseline_int.csv')

od_merged = od_actual[['o','d','人数']].merge(
    od_baseline[['o','d','人数']].rename(columns={'人数': '人数_baseline'}),
    on=['o','d'], how='outer'
).fillna(0)
od_merged['人数_diff'] = od_merged['人数'] - od_merged['人数_baseline']
print('od_merged shape:', od_merged.shape)
print(od_merged['人数_diff'].describe())


In [ ]:
def create_flowline_v1(
    df_od: pd.DataFrame,
    fence: gpd.GeoDataFrame,
    output_path,
    flow_col: str = '人数',
    o_col: str = 'o',
    d_col: str = 'd',
    top_n: int = 500,
    is_diff: bool = False,
    cmap_single: str = 'YlOrRd',
    linewidth_min: float = 0.5,
    linewidth_max: float = 8.0,
    alpha_min: float = 0.2,
    alpha_max: float = 0.85,
) -> None:
    """
    OD 流线图 v1：基于参考代码逐条绘制，颜色/线宽/透明度均按流量映射。
    线宽图例固定在左下角，无图题。
    """
    # 投影到平面坐标系，获取 TAZ 中心点
    fence_proj = fence.copy()
    if fence_proj.crs and fence_proj.crs.is_geographic:
        fence_proj = fence_proj.to_crs('EPSG:32649')
    centroids = fence_proj.geometry.centroid
    taz_col = fence.columns[0]
    coord_dict = {tid: (c.x, c.y) for tid, c in zip(fence[taz_col].values, centroids)}

    df = df_od[[o_col, d_col, flow_col]].copy().dropna()
    df = df[df[o_col].isin(coord_dict) & df[d_col].isin(coord_dict)]
    df = df.reindex(df[flow_col].abs().nlargest(top_n).index)

    if df.empty:
        print('无有效 OD 数据，跳过')
        return

    # 按流量绝对值从小到大排序（小的先画，大的后画，大的在最上层）
    df = df.reindex(df[flow_col].abs().sort_values().index)

    flows = df[flow_col].values
    abs_flows = np.abs(flows)
    flow_min, flow_max = abs_flows.min(), abs_flows.max()
    if flow_max == flow_min:
        flow_max = flow_min + 1

    # 颜色映射
    if is_diff:
        abs_max = float(abs_flows.max()) or 1
        norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
        cmap = mpl.cm.get_cmap('RdBu_r')
        color_vals = flows
    else:
        norm = mcolors.Normalize(vmin=flow_min, vmax=flow_max)
        cmap = mpl.cm.get_cmap(cmap_single)
        color_vals = abs_flows

    # 线宽映射
    lw_arr = linewidth_min + (abs_flows - flow_min) / (flow_max - flow_min) * (linewidth_max - linewidth_min)
    # 透明度映射
    alpha_arr = alpha_min + (abs_flows - flow_min) / (flow_max - flow_min) * (alpha_max - alpha_min)

    fig, ax = plt.subplots(figsize=VISUAL_CONFIG['figure_size'])

    # 底图
    fence_proj.plot(ax=ax, color='#F5F5F5', edgecolor='#CCCCCC', linewidth=0.3)
    boundary = gpd.GeoSeries([unary_union(fence_proj.geometry)])
    boundary.boundary.plot(ax=ax, color='#333333', linewidth=1.5)

    # 逐条绘制 OD 流线（参考代码风格：for 循环，小流量先画）
    for i, (_, row) in enumerate(df.iterrows()):
        o_xy = coord_dict[row[o_col]]
        d_xy = coord_dict[row[d_col]]
        color_i = cmap(norm(color_vals[i]))
        ax.plot(
            [o_xy[0], d_xy[0]], [o_xy[1], d_xy[1]],
            color=color_i,
            linewidth=lw_arr[i],
            alpha=float(alpha_arr[i]),
            solid_capstyle='round',
        )

    # colorbar（颜色图例）
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.5, aspect=20, pad=0.02)
    cbar.set_label(flow_col, fontsize=VISUAL_CONFIG['label_fontsize'], fontweight='bold')
    cbar.ax.tick_params(labelsize=VISUAL_CONFIG['legend_fontsize'])

    # 线宽图例（固定左下角）
    lw_bins = [flow_min, (flow_min + flow_max) / 2, flow_max]
    lw_labels = [f'{int(v)}' for v in lw_bins]
    lw_widths = [linewidth_min, (linewidth_min + linewidth_max) / 2, linewidth_max]
    lw_legend_elements = [
        plt.Line2D([0], [0], color='black', linewidth=w, label=l)
        for l, w in zip(lw_labels, lw_widths)
    ]
    ax.legend(handles=lw_legend_elements, loc='lower left',
              bbox_to_anchor=(0.0, 0.0),
              title=f'{flow_col} (线宽)',
              fontsize=VISUAL_CONFIG['legend_fontsize'],
              title_fontsize=VISUAL_CONFIG['label_fontsize'],
              frameon=True, fancybox=True, shadow=True,
              edgecolor='#CCCCCC', facecolor='white', framealpha=0.95)

    ax.set_axis_off()
    _add_scalebar_auto(ax, fence)
    add_north_arrow(ax)
    plt.tight_layout()

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=VISUAL_CONFIG['dpi'],
                bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'saved: {output_path}')


In [ ]:
# 输出差值流线图 v1
create_flowline_v1(
    df_od=od_merged,
    fence=fence,
    output_path=OUT_DIR / 'v1_flowline_diff.png',
    flow_col='人数_diff',
    o_col='o',
    d_col='d',
    top_n=500,
    is_diff=True,
    linewidth_min=0.5,
    linewidth_max=8.0,
    alpha_min=0.2,
    alpha_max=0.85,
)
from IPython.display import Image
Image(str(OUT_DIR / 'v1_flowline_diff.png'))
